---

# Baixa os Dados de Flashes Diretamento do FTP da Amazon do Sensor GLM do Satelite GOES-16 ou GOES-19 para um Determinado Dia

---

- `OBJETIVO`:
> Este código baixa os dados de flashes do sensor GLM do satélite GOES-16 ou GOES-19 do repositório da AMAZON. Os dados são baixados no Google Drive e armazenados na pasta `glm_20s_goes16` ou `glm_20s_goes19`. Dentro desta pasta existe o subdiretório daquele dia e uma pasta para cada horário do dia (0, 1,...23h). Exemplo: dados do dia 30 de junho de 2020 ás 15 horas UTC: `output/glm_20s_goes16/2020-06-290/15/`.

- `DADOS`:
> Dados do sensor GLM do satélite [GOES-16](https://noaa-goes16.s3.amazonaws.com/index.html#GLM-L2-LCFA/) ou [GOES-19](https://noaa-goes19.s3.amazonaws.com/index.html#GLM-L2-LCFA/) fornecido pela AMAZON. Exemplo de nome do arquivo: `OR_GLM-L2-LCFA_G16_s20201820000000_e20201820000200_c20201820000224.nc`

- `OBSERVAÇÕES`:
    > Em **1 dia** de dados temos `4320 arquivos` e demora `1h38s` para baixar.

- `REALIZADO POR`:
> Enrique V. Mattos - 03/10/2025

- `ATUALIZADO POR`:
> Enrique V. Mattos - 27/04/2026
---



# **1° Passo:** Preparando ambiente

In [ ]:
# instala e importa bibliotecas
!pip install -q boto3

# importa bibliotecas
import os
import time
from datetime import timedelta, datetime

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA'

# diretório de saída
dir_output = f'{dir}/output/glm_20s_goes'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)

# **2° Passo:** Declarando funções

In [ ]:
#==================================================================================================#
#                   FUNÇÃO QUE FAZ O DOWNLOAD DOS DADOS DE 20S DA AWS
#==================================================================================================#
def download_GLM20s_AWS(yyyymmddhhmnss, goes_number, path_dest):

    ''' Função para baixar os arquivos netcdf de 20s do GLM processado e disponibilizado pela Amazon Web Service (AWS)

    Parâmetros:
               yyyymmddhhmnss (string): ano+mes+dia+hora+minuto+segundo. Exemplo: 20251113033020
               goes_number (string): número do satélite GOES: 16 ou 19
               path_dest (string): nome do direório + nome do arquivo que foi baixado

    Retorna:
            file_name (string): nome do direório + nome do arquivo que foi baixado

    Observação: acesso aos dados
                    GOES-16: https://noaa-goes16.s3.amazonaws.com/index.html#ABI-L2-CMIPF/
                    GOES-19: https://noaa-goes19.s3.amazonaws.com/index.html#ABI-L2-CMIPF/
    '''

    year = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%Y')
    day_of_year = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%j')
    hour = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%H')
    min = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%M')
    seg = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%S')

    # informação do repositório da AMAZON. Exemplo: 'noaa-goes19'
    bucket_name = f'noaa-goes{goes_number}'

    # inicializa o S3 cliente
    s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

    # estutura do arquivo. Exemplo: "GLM-L2-LCFA/2025/317/03/OR_GLM-L2-LCFA_G19_s20253170330200"
    product_name = "GLM-L2-LCFA"
    prefix = f'{product_name}/{year}/{day_of_year}/{hour}/OR_{product_name}_G{goes_number}_s{year}{day_of_year}{hour}{min}{seg}'

    # pesquisa pelo arquivo no servidor
    s3_result = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix, Delimiter = "/")

    # verifica se o arquivo esta diponível
    #-----------------------------------------------------------------------------------------------------------
    if 'Contents' not in s3_result:

        # se o arquivo não esta disponível
        print(f'No files found for the date: {yyyymmddhhmnss}, Product-{product_name}')
        return -1
    else:

        # se o arquivo existe
        for obj in s3_result['Contents']:
            key = obj['Key']

            # imprime o nome do arquivo
            file_name = key.split('/')[-1].split('.')[0]

        # baixa o arquivo
        if os.path.exists(f'{path_dest}/{file_name}.nc'):
            print(f'File {path_dest}/{file_name}.nc exists')
        else:
            print(f'Downloading file {path_dest}/{file_name}.nc')
            s3_client.download_file(bucket_name, key, f'{path_dest}/{file_name}.nc')

    return f'{file_name}'

# **Processando os dados**

In [ ]:
%%time
#========================================================================================================================#
#                                          IMPORTAÇÃO DAS BIBLIOTECAS
#========================================================================================================================#
from datetime import timedelta, datetime
import os
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#========================================================================================================================#
#                                          DEFINE A DATA DO ARQUIVO
#========================================================================================================================#
# data INICIAL
anoi, mesi, diai, hori, mini, segi = '2020', '06', '30', '00', '00', '00'

# data FINAL
anof, mesf, diaf, horf, minf, segf = '2020', '06', '30', '23', '59', '59'

#========================================================================================================================#
#                                          CRIAS AS PASTAS POR HORÁRIO
#========================================================================================================================#
# cria as pastas de saida: ano-mes-dia/hora
for data in pd.date_range(f'{anoi}-{mesi}-{diai} {hori}:{mini}', f'{anof}-{mesf}-{diaf} {horf}:{minf}', freq='1H'):

    # extrai ano e mês
    anox = data.strftime('%Y')
    mesx = data.strftime('%m')
    diax = data.strftime('%d')
    horx = data.strftime('%H')

    # cria pasta daquele horário
    os.makedirs(f'{dir_output}/{anox}-{mesx}-{diax}/{horx}', exist_ok=True)

#========================================================================================================================#
#                                                DOWNLOAD DOS DADOS
#========================================================================================================================#
# data INICIAL junta como uma string
date_ini = str(datetime(int(anoi),int(mesi),int(diai),int(hori),int(mini)))

# data FINAL junta como uma string
date_end = str(datetime(int(anof),int(mesf),int(diaf),int(horf),int(minf)))

# data inicial é fixada como a data do loop
date_loop = date_ini

# loop nos arquivos do GLM
while (date_loop <= date_end):

    # data
    yyyymmddhhmnss = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y%m%d%H%M%S')

    # ano, mes, dia, hora, minuto e segundos
    ano = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y')
    mes = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%m')
    dia = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%d')
    hor = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%H')
    min = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%M')
    seg = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%S')

    # define se GOES-16 ou GOES-19
    start_g19 = datetime(2025,4,7,0,0)
    imagem_atual = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S')
    goes_number = '16' if imagem_atual < start_g19 else '19'
    print(f'DOWNLOAD DO ARQUIVO GLM ===>>> {ano}-{mes}-{dia} {hor}:{min}:{seg}')

    # local onser será baixado o arquivo
    local_salvar_arquivo = f'{dir_output}/{ano}-{mes}-{dia}/{hor}'

    # download o arquivo
    file_glm20s = download_GLM20s_AWS(yyyymmddhhmnss, goes_number, local_salvar_arquivo)

    # incrementa a variável the date_loop
    date_loop = str(datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S') + timedelta(seconds=20))
    print('\n')

In [ ]:
# lista os arquivos baixados
import glob
files = sorted(glob.glob(f'{dir_output}/2020-06-29/*/*.nc'))
files

In [ ]:
# quantidade de arquivos
len(files)